# 08 · Điểm mix học bằng InfoNCE (ticket #23)

Học hai MLP `f_out` (bài đang phát) và `f_in` (bài kế tiếp) trên đặc trưng âm thanh giống luật (BPM, tông Camelot, energy), từ các cặp bài liền nhau trong **phiên train**. Mẫu âm là các bài kế tiếp khác trong cùng batch, nhiệt độ τ = 0,1. Dừng sớm theo loss InfoNCE trên cặp val (cặp val chỉ để theo dõi, không học). `s_learn(a, b) = f_out(x_a)ᵀ f_in(x_b)` (đầu ra chuẩn hóa L2, nên trong [−1, 1]); điểm mix đưa vào rerank là `(1 + s_learn) / 2` trong 0–1.

**Input** (Add Input): Kaggle Dataset `titeohihi/mixnext-splited` (output của notebook 03).

**Output** (`/kaggle/working/out/`):
- `mix_learned.npz`: trọng số numpy của `f_out`, `f_in` cho `LearnedMix.load` (app không cần torch)
- `mix_learned.json`: tham số, số cặp train/val, loss và Recall@10 trong batch trên cặp val theo từng epoch

Bật **Internet** để clone repo. Chỉ cần CPU: khoảng 2–3 phút mỗi epoch với khoảng 5 triệu cặp train, giới hạn 30 phút (`MixTrainParams.minutes`).

In [ ]:
# Bootstrap: lấy code từ GitHub. Khi #23 đã merge thì đổi BRANCH thành "main".
BRANCH = "a"
REPO = "/kaggle/working/MusicRecSys"
!rm -rf {REPO} && git clone -q --depth 1 -b {BRANCH} https://github.com/yashita23/MusicRecSys {REPO}
!pip install -q --no-deps -e {REPO}
import sys; sys.path.insert(0, REPO)

In [ ]:
from pathlib import Path
DATA = sorted(Path("/kaggle/input").rglob("sessions.parquet"))[0].parent
OUT = Path("/kaggle/working/out")
print("input:", DATA, sorted(p.name for p in DATA.iterdir()))

In [ ]:
import time
from src.rerank.mix_learned_train import main

t0 = time.time()
report = main(["--data", str(DATA), "--out", str(OUT)])   # học trên cặp train, dừng sớm theo loss trên cặp val
print(f"{time.time() - t0:.0f} giây")
for p in sorted(OUT.rglob("*.*")):
    print(p.relative_to(OUT), f"{p.stat().st_size / 1e3:.1f} KB")

In [ ]:
# Tiêu chí hoàn thành của #23: loss val giảm và ổn định; Recall@10 trong batch val hơn ngẫu nhiên; chỉ học từ cặp train
import pandas as pd

history = pd.DataFrame(report["history"])
print(history.round(4).to_string(index=False))
best = report["val"]
print(f"\nCặp train {report['n_train_pairs']:,}, cặp val {report['n_val_pairs']:,}; epoch tốt nhất {report['best_epoch']}")
print(f"loss val: chưa học {history['val_loss'].iloc[0]:.4f} → {best['val_loss']:.4f}")
print(f"Recall@10 trong batch val: {best['val_recall_at_10']:.4f} (ngẫu nhiên {best['random_recall_at_10']:.4f}, gấp {best['val_recall_at_10'] / best['random_recall_at_10']:.1f} lần)")

In [ ]:
# Câu hỏi phụ (spec §6): điểm mix học có đồng ý với luật DJ không? Spearman trên các cặp bài ngẫu nhiên.
import numpy as np
from src.rerank.mix_learned import LearnedMix
from src.rerank.mix_rule import RuleMix

tracks = pd.read_parquet(DATA / "tracks.parquet")
learned, rule = LearnedMix.load(tracks, OUT / "mix_learned.npz"), RuleMix(tracks)
rng = np.random.default_rng(0)
ids = tracks["track_id"].to_numpy()
rows = []
for a in rng.choice(ids, 1000, replace=False):
    cands = list(rng.choice(ids, 100, replace=False))
    rows.append(pd.DataFrame({"learned": learned(a, cands), "rule": rule(a, cands)}))
pairs = pd.concat(rows, ignore_index=True)
print(f"Spearman(điểm mix học, điểm mix theo luật) trên {len(pairs):,} cặp: {pairs.corr(method='spearman').iloc[0, 1]:.3f}")
# Độ trải của điểm: điểm mix học co cụm hơn luật thì cùng một α sẽ cân phần mix nhẹ hơn (xem trước khi chọn α, #25)
print(pairs.describe(percentiles=[0.1, 0.5, 0.9]).round(3).to_string())

## Đăng output

1. **Save Version → Save & Run All (Commit)** để lưu output.
2. Tab Output của version vừa chạy → **New Dataset** tên `mixnext-mix-learned`, chỉ giữ `out/`. App đọc `out/mix_learned.npz` sau khi chép vào `artifacts/` (#20).

## Kết quả (chạy trên Kaggle, …, ngày)

- …